# SSS Marine Debris Detection — Model Training

## 6 Model Variants
| Variant | Name | Backbone | Training |
|---------|------|----------|----------|
| A | YOLOv8n baseline | COCO pretrained | Fine-tune |
| B | YOLOv8s baseline | COCO pretrained | Fine-tune |
| C | SS-YOLO scratch | Random init | From scratch |
| D | SS-YOLO pretrained | COCO pretrained | Fine-tune |
| E | SS-YOLO+EIS scratch | Random init | From scratch |
| F | SS-YOLO pretrained+EIS | COCO pretrained | Fine-tune |

## Dataset: H8 (F6 + G7)
- 3110 train images (561 debris + 2549 BG)
- 438 val images
- Hard negatives from raw TIFFs

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1: Setup
# ═══════════════════════════════════════════════════════════
!pip install ultralytics pandas matplotlib -q
!git clone https://github.com/Dinoman67/sonarvision.git
%cd sonarvision

import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2: Upload Dataset
# ═══════════════════════════════════════════════════════════
# Upload h8.zip from your machine
# File location: /tmp/h8.zip (752MB)
from google.colab import files
uploaded = files.upload()  # Select h8.zip

# Unzip
!unzip -q h8.zip -d /content/

# Fix data.yaml path for Colab
!sed -i 's|path:.*|path: /content/h8|' /content/h8/data.yaml

DATA = '/content/h8/data.yaml'
print(f'\nDataset ready: {DATA}')

# Verify
import yaml
from pathlib import Path
with open(DATA) as f:
    cfg = yaml.safe_load(f)
train_imgs = list(Path(cfg['path'], 'images', 'train').glob('*'))
val_imgs = list(Path(cfg['path'], 'images', 'val').glob('*'))
print(f'Train: {len(train_imgs)} images')
print(f'Val: {len(val_imgs)} images')
print(f'Classes: {cfg["names"]}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3: Load Custom Modules
# ═══════════════════════════════════════════════════════════
import sys
from ultralytics import YOLO
from models.sss_custom_modules import (
    PConv, FasterBlock, FastC2f, GhostConv,
    SEBlock, CBAM, WaveletConv, LocalContrastEnhance
)
from models.build_sss_models import (
    build_model, build_model_a, build_model_b,
    build_model_c, build_model_d, build_model_e, build_model_f,
    build_yolov8_esi_full, C2fWithSE,
    print_model_comparison
)
from ultralytics.models.yolo.detect.train import DetectionTrainer

print_model_comparison()
print('\n✓ All custom modules loaded')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4: Training Helpers
# ═══════════════════════════════════════════════════════════
import pandas as pd

def evaluate(model_path, data_yaml, imgsz=256, conf=0.05):
    """Evaluate model and return metrics."""
    m = YOLO(model_path)
    r = m.val(data=data_yaml, imgsz=imgsz, conf=conf, verbose=False)
    p, rv = r.box.mp, r.box.mr
    f1 = 2*p*rv / max(p+rv, 1e-8)
    return {'mAP50': r.box.map50, 'P': p, 'R': rv, 'F1': f1}

def conf_sweep(model_path, data_yaml, imgsz=256):
    """Find optimal confidence threshold."""
    m = YOLO(model_path)
    best_f1, best_conf = 0, 0.05
    for c in [0.01, 0.02, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3]:
        r = m.val(data=data_yaml, imgsz=imgsz, conf=c, verbose=False)
        p, rv = r.box.mp, r.box.mr
        f1 = 2*p*rv / max(p+rv, 1e-8)
        if f1 > best_f1:
            best_f1, best_conf = f1, c
    return best_conf, best_f1

def patch_trainer(model_obj):
    """Patch DetectionTrainer to use custom model."""
    _orig = DetectionTrainer.get_model
    def _patched(self, cfg=None, weights=None, verbose=True):
        from ultralytics.nn.tasks import DetectionModel
        from ultralytics.utils import RANK
        dm = DetectionModel(cfg, nc=self.data['nc'], ch=self.data['channels'],
                            verbose=verbose and RANK == -1)
        dm.model = model_obj.model
        dm.nc = 1
        dm.names = {0: 'marine_debris'}
        try:
            dm.load(weights)
        except Exception:
            pass
        return dm
    DetectionTrainer.get_model = _patched
    return _orig

print('✓ Helpers loaded')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5: Stage 1 — Train YOLOv8n (Model A)
# ═══════════════════════════════════════════════════════════
print('='*60)
print('STAGE 1: Model A — YOLOv8n baseline')
print('='*60)

model_a = YOLO('yolov8n.pt')
model_a.train(
    data=DATA, epochs=30, imgsz=256, batch=32, patience=15,
    lr0=0.01, lrf=0.01, warmup_epochs=2,
    mosaic=0.0, mixup=0.0,
    fliplr=0.0, flipud=0.0, degrees=0.0,
    translate=0.05, scale=0.2,
    name='model_a_yolov8n', project='/content/runs', exist_ok=True, plots=True,
)
print('✓ Model A done')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6: Stage 1 — Train SS-YOLO (Model D — pretrained)
# ═══════════════════════════════════════════════════════════
print('='*60)
print('STAGE 1: Model D — SS-YOLO pretrained backbone')
print('='*60)

model_d_obj = build_model_d()
_orig = patch_trainer(model_d_obj)

try:
    yolo_d = YOLO('yolov8n.pt')
    yolo_d.train(
        data=DATA, epochs=30, imgsz=256, batch=32, patience=15,
        lr0=0.01, lrf=0.01, warmup_epochs=2,
        freeze=10,
        mosaic=0.0, mixup=0.0,
        fliplr=0.0, flipud=0.0, degrees=0.0,
        translate=0.05, scale=0.2,
        name='model_d_ss_yolo', project='/content/runs', exist_ok=True, plots=True,
    )
finally:
    DetectionTrainer.get_model = _orig

print('✓ Model D done')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7: Stage 1 — Train YOLOv8-ESI (SE attention)
# ═══════════════════════════════════════════════════════════
print('='*60)
print('STAGE 1: YOLOv8-ESI (SE attention)')
print('='*60)

esi_obj = build_yolov8_esi_full()
_orig2 = patch_trainer(esi_obj)

try:
    yolo_esi = YOLO('yolov8n.pt')
    yolo_esi.train(
        data=DATA, epochs=30, imgsz=256, batch=32, patience=15,
        lr0=0.01, lrf=0.01, warmup_epochs=2,
        mosaic=0.0, mixup=0.0,
        fliplr=0.0, flipud=0.0, degrees=0.0,
        translate=0.05, scale=0.2,
        name='model_esi', project='/content/runs', exist_ok=True, plots=True,
    )
finally:
    DetectionTrainer.get_model = _orig2

print('✓ YOLOv8-ESI done')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 8: Stage 1 — Compare All Models
# ═══════════════════════════════════════════════════════════
print('='*60)
print('STAGE 1: COMPARISON RESULTS')
print('='*60)

models_to_eval = [
    ('YOLOv8n', '/content/runs/model_a_yolov8n/weights/best.pt'),
    ('SS-YOLO-D', '/content/runs/model_d_ss_yolo/weights/best.pt'),
    ('YOLOv8-ESI', '/content/runs/model_esi/weights/best.pt'),
]

results = []
for name, path in models_to_eval:
    try:
        r = evaluate(path, DATA, conf=0.05)
        results.append({'Model': name, **r})
        print(f'  {name}: mAP50={r["mAP50"]:.4f}, P={r["P"]:.4f}, R={r["R"]:.4f}, F1={r["F1"]:.4f}')
    except Exception as e:
        print(f'  {name}: FAILED — {e}')

df = pd.DataFrame(results)
print('\n' + df.to_string(index=False))

if len(results) > 0:
    winner = max(results, key=lambda x: x['F1'])
    print(f'\n🏆 WINNER: {winner["Model"]} (F1={winner["F1"]:.4f})')
    winner_name = winner['Model']

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 9: Stage 2 — Train Winner
# ═══════════════════════════════════════════════════════════
print('='*60)
print(f'STAGE 2: Training {winner_name} (150 epochs)')
print('='*60)

if winner_name == 'YOLOv8n':
    model_s2 = YOLO('yolov8n.pt')
    model_s2.train(
        data=DATA, epochs=150, imgsz=256, batch=16, patience=30,
        lr0=0.001, lrf=0.01, warmup_epochs=3,
        freeze=10,
        mosaic=0.0, mixup=0.0,
        fliplr=0.0, flipud=0.0, degrees=0.0,
        translate=0.05, scale=0.2,
        name='yolov8n_s2_final', project='/content/runs', exist_ok=True, plots=True,
    )
elif winner_name == 'SS-YOLO-D':
    model_d_obj = build_model_d()
    _orig = patch_trainer(model_d_obj)
    try:
        yolo_d = YOLO('yolov8n.pt')
        yolo_d.train(
            data=DATA, epochs=200, imgsz=256, batch=16, patience=40,
            lr0=0.001, lrf=0.01, warmup_epochs=5,
            freeze=10,
            mosaic=0.0, mixup=0.0,
            fliplr=0.0, flipud=0.0, degrees=0.0,
            translate=0.05, scale=0.2,
            name='ss_yolo_s2_final', project='/content/runs', exist_ok=True, plots=True,
        )
    finally:
        DetectionTrainer.get_model = _orig
elif winner_name == 'YOLOv8-ESI':
    esi_obj = build_yolov8_esi_full()
    _orig = patch_trainer(esi_obj)
    try:
        yolo_esi = YOLO('yolov8n.pt')
        yolo_esi.train(
            data=DATA, epochs=150, imgsz=256, batch=16, patience=30,
            lr0=0.001, lrf=0.01, warmup_epochs=3,
            freeze=10,
            mosaic=0.0, mixup=0.0,
            fliplr=0.0, flipud=0.0, degrees=0.0,
            translate=0.05, scale=0.2,
            name='esi_s2_final', project='/content/runs', exist_ok=True, plots=True,
        )
    finally:
        DetectionTrainer.get_model = _orig

print(f'\n✓ Stage 2 complete for {winner_name}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 10: Final Evaluation + Conf Sweep
# ═══════════════════════════════════════════════════════════
print('='*60)
print('FINAL EVALUATION')
print('='*60)

import glob
weight_dirs = glob.glob('/content/runs/*_s2_final/weights/best.pt')
if weight_dirs:
    best_weights = weight_dirs[0]
else:
    weight_dirs = glob.glob('/content/runs/*/weights/best.pt')
    best_weights = weight_dirs[-1] if weight_dirs else None

if best_weights:
    print(f'Using weights: {best_weights}')
    
    best_conf, best_f1 = conf_sweep(best_weights, DATA)
    print(f'Best conf threshold: {best_conf} (F1={best_f1:.4f})')
    
    final = evaluate(best_weights, DATA, conf=best_conf)
    print(f'\nFinal metrics (conf={best_conf}):')
    print(f'  mAP50: {final["mAP50"]:.4f}')
    print(f'  Precision: {final["P"]:.4f}')
    print(f'  Recall: {final["R"]:.4f}')
    print(f'  F1: {final["F1"]:.4f}')
else:
    print('No trained weights found!')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 11: Export
# ═══════════════════════════════════════════════════════════
if best_weights:
    print('Exporting to ONNX...')
    export_model = YOLO(best_weights)
    export_model.export(format='onnx', imgsz=256)
    print('✓ Exported!')
    
    from google.colab import files
    files.download(best_weights)
    print('\nDownload started!')